# Clinical Validation of 2D Mammography AI — CBIS-DDSM

A reproducible **standalone (black-box) validation** of a pretrained breast-cancer classifier on the CBIS-DDSM test split.

The pipeline never trains or modifies the model — it *measures* it against a biopsy-proven reference standard, which is what clinical validation (as opposed to model development) is.

**Dataset:** CBIS-DDSM (Curated Breast Imaging Subset of DDSM), Kaggle JPEG mirror — biopsy-proven pathology, BI-RADS assessment, breast density, lesion type.
**Model:** a HuggingFace Swin-V2 breast classifier (a *baseline* — swap in any stronger model by writing one `ImageClassifier`).
**Output:** an HTML validation report — ROC/AUC with DeLong CIs, operating points, calibration, screening behaviour, breast-density subgroup analysis, and an AI-vs-radiologist (BI-RADS) comparison.

## 1 · Environment setup
Runs on **Google Colab** or **Kaggle**.

**On Kaggle:** in the right-hand panel set *Accelerator* to a GPU and turn *Internet* **On** (needed to `pip install transformers` and download the model from the HuggingFace Hub).

Get the `mammoval` package onto the machine first — either enable Internet and `!git clone` your repo, or add the project folder as a dataset via *+ Add Input*. Then run the cells below.

In [ ]:
!pip install -q transformers timm pillow
# torch / torchvision / pydicom are pre-installed on Colab and Kaggle.

In [ ]:
# Locate the mammoval package. Works on Colab and Kaggle, whether you
# git-cloned the repo or added it as a Kaggle dataset / uploaded the folder.
import os, sys, glob
cands = ['.', 'breast-ai-clinical-validation', '/content/breast-ai-clinical-validation']
cands += glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
cands += glob.glob('/kaggle/working/*')
for c in cands:
    if c and os.path.isdir(os.path.join(c, 'mammoval')):
        sys.path.insert(0, os.path.abspath(c)); print('found mammoval in', c); break
else:
    raise RuntimeError(
        'mammoval package not found. Either (a) enable Internet and run '
        '!git clone <your-repo-url>, or (b) add the project folder as a '
        'Kaggle dataset via "+ Add Input".')
import mammoval
print('mammoval', mammoval.__version__, 'ready')

## 2 · Get the CBIS-DDSM dataset
**On Kaggle (easiest):** click **+ Add Input**, search *"CBIS-DDSM Breast Cancer Image Dataset"* (by *awsaf49*) and add it — it mounts read-only, no download or credentials needed.

**On Colab:** the cell downloads the ~6 GB JPEG mirror; it will prompt you to upload `kaggle.json` (Kaggle → Settings → Create New API Token).

In [ ]:
import os
ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    CBIS_ROOT = '/kaggle/input/cbis-ddsm-breast-cancer-image-dataset'
    assert os.path.isdir(CBIS_ROOT), (
        'Add the awsaf49 "CBIS-DDSM Breast Cancer Image Dataset" '
        'via + Add Input first.')
else:
    from google.colab import files
    print('Upload your kaggle.json:'); files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip install -q kaggle
    !kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset
    !unzip -q -o cbis-ddsm-breast-cancer-image-dataset.zip -d data/cbis-ddsm
    CBIS_ROOT = 'data/cbis-ddsm'
print('CBIS-DDSM root:', CBIS_ROOT)

## 3 · Load the dataset into a standard case table
`CBISDDSMDataset` joins the case-description CSVs to the JPEG mirror and aggregates abnormality rows to one image-level case (`y_true` = malignant if any abnormality is malignant).

In [ ]:
from mammoval.data import CBISDDSMDataset
import json

ds = CBISDDSMDataset(root=CBIS_ROOT, split='test')
print(json.dumps(ds.summary(), indent=2, default=str))
ds.cases.head()

## 4 · Load the pretrained model — treated as a black box
`HFImageClassifier` wraps a HuggingFace checkpoint and auto-detects which class index is malignant. The default is a verified Swin-V2 breast classifier. **This is a baseline**: it was fine-tuned on Mini-DDSM-derived images, so expect domain shift on CBIS-DDSM — the report will quantify exactly that.

In [ ]:
from mammoval.models import HFImageClassifier

model = HFImageClassifier()    # or HFImageClassifier('your/model-id')
print('model :', model.name)
print('labels:', model.id2label, '| malignant index =', model.malignant_index)

## 5 · Run inference → predictions table
`score_dataset` runs the model over every case and returns the case table with a `y_score` column. Set `LIMIT=None` for the full test split; a few hundred cases is enough for a quick pass.

In [ ]:
from mammoval.models import score_dataset

LIMIT = 400          # set to None for the whole test split
preds = score_dataset(model, ds, limit=LIMIT)
preds = preds.dropna(subset=['y_score'])
preds[['case_id','y_true','y_score','breast_density_cat',
       'abnormality_type','birads_assessment']].head()

## 6 · Run the clinical validation pipeline
One call produces the full result: discrimination, operating points, calibration, screening behaviour, subgroup analysis, and the paired AI-vs-reader comparison (BI-RADS assessment is used as the radiologist reference).

In [ ]:
from mammoval.pipeline import ClassificationConfig, run_classification_validation

cfg = ClassificationConfig(
    reader_col='birads_assessment',
    subgroup_cols=('breast_density_cat', 'abnormality_type', 'view'),
    target_specificities=(0.90, 0.96),
    ni_margin=0.05,
    is_probability=True,
    dataset_name='CBIS-DDSM test split',
)
results = run_classification_validation(preds, cfg)
d = results['discrimination']
print('AUC = %.3f   95%% CI %.3f-%.3f' % (d['auc'], *d['auc_ci']))

## 7 · Generate the HTML validation report

In [ ]:
from mammoval.report import build_report
from IPython.display import HTML

os.makedirs('outputs', exist_ok=True)
path = build_report(
    results,
    output_path='outputs/cbis_ddsm_validation_report.html',
    title='2D Mammography AI — Clinical Validation (CBIS-DDSM)',
    extra_limitations=[
        'CBIS-DDSM is digitised film, not full-field digital mammography — '
        'a real domain shift from a modern screening device input.',
        'The default Swin-V2 model is a baseline trained on Mini-DDSM-'
        'derived images; treat its absolute numbers as a lower bound.',
    ])
HTML(open(path, encoding='utf-8').read())

## Notes & next steps
* **Stronger model.** Replace `HFImageClassifier()` with the RSNA-2023 winning ensemble or a Mammo-CLIP linear probe — only the adapter changes; the validation is identical.
* **Reader caveat.** BI-RADS *assessment* is an ordinal proxy reader, not an independent radiologist read; category 0 ('incomplete') is not strictly ordered. A true reader study is MRMC (see `docs/methodology.md`).
* **External validation.** Re-run on VinDr-Mammo or RSNA to probe generalisation across vendor / population — the report's CIs cover sampling error only, not distribution shift.